# MIMIC-IV Stage IV / M1 Coverage Search

**Purpose:** Count viable Stage IV / M1 NSCLC patients in the 1,527-patient MIMIC-IV cohort (patients with both discharge and radiology notes) to inform supervisor feedback Item 6.

**Decision to make:**
- **Path A** — Scope Guideline Logic Agent evaluation to T/N staging only (document Stage IV as out-of-scope). Low effort, honest about data limits.
- **Path B** — Supplement NSCLC-Radiomics with MIMIC-IV M1 cases for full TNM staging validation. Requires enough high-confidence Stage IV / M1 patients here.

**Cohort source:** `data/processed/cohort/cohort_patients.csv` (2,283 patients), filtered to those with both discharge and radiology notes in MIMIC-IV-Note v2.2.

## Imports

In [ ]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
random.seed(42)

DATA_DIR      = Path('data')
PROCESSED_DIR = DATA_DIR / 'processed' / 'cohort'
NOTE_DIR      = DATA_DIR / 'raw' / 'mimic-iv-note-2.2' / 'note'

## Load cohort and notes

In [ ]:
cohort_subjects = pd.read_csv(PROCESSED_DIR / 'cohort_patients.csv')['subject_id'].unique()
print(f'Cohort size: {len(cohort_subjects):,}')

# Load full text columns — filtered to cohort subjects via chunked read
# (discharge.csv.gz = 1.1 GB compressed; radiology.csv.gz = 746 MB compressed;
#  reading in 50k-row chunks avoids loading all of MIMIC into memory)
cohort_set = set(cohort_subjects)

def load_filtered(path, cohort_set, chunk=50_000):
    frames = []
    for c in pd.read_csv(path, compression='gzip',
                          usecols=['note_id', 'subject_id', 'text'],
                          chunksize=chunk):
        hit = c[c['subject_id'].isin(cohort_set)]
        if len(hit):
            frames.append(hit)
    return pd.concat(frames, ignore_index=True)

print('Loading discharge notes...')
discharge_df = load_filtered(NOTE_DIR / 'discharge.csv.gz', cohort_set)
print(f'  {len(discharge_df):,} notes for {discharge_df["subject_id"].nunique():,} patients')

print('Loading radiology notes...')
radiology_df = load_filtered(NOTE_DIR / 'radiology.csv.gz', cohort_set)
print(f'  {len(radiology_df):,} notes for {radiology_df["subject_id"].nunique():,} patients')

# Combine and group all note text per patient
all_notes = pd.concat([discharge_df, radiology_df], ignore_index=True)
patient_text = (
    all_notes.groupby('subject_id')['text']
    .apply(lambda s: ' '.join(s.fillna('')))
    .reset_index()
)

# Restrict to patients with BOTH discharge and radiology notes (the 1,527 cohort)
ds_ids = set(discharge_df['subject_id'])
rr_ids = set(radiology_df['subject_id'])
both_ids = ds_ids & rr_ids
patient_text = patient_text[patient_text['subject_id'].isin(both_ids)].reset_index(drop=True)
print(f'\nSearch cohort (both note types): {len(patient_text):,} patients')

## Define search patterns

Case-insensitive regex patterns covering Stage IV, M1, and explicit metastasis terminology. A negation check inspects the 100 characters *before* each match for phrases like "no ", "without ", "negative for " — matches preceded by negation are flagged as likely false positives rather than discarded, so we can verify them in the sample cells.

In [ ]:
stage_iv_pattern = re.compile(
    r'\b(stage\s*(iv|4|four))\b', re.IGNORECASE
)

# M1 intentionally case-sensitive: M1 is the TNM symbol;
# lowercase m1 appears in immunology contexts (M1 macrophages)
m1_pattern = re.compile(r'\bM1[abc]?\b')

mets_pattern = re.compile(
    r'(metastatic\s+disease'
    r'|metastases?\s+to\s+(brain|bone|liver|adrenal|pleura)'
    r'|distant\s+metastas[ie]s'
    r'|brain\s+metastas[ie]s'
    r'|bone\s+metastas[ie]s'
    r'|liver\s+metastas[ie]s)',
    re.IGNORECASE
)

negation_pattern = re.compile(
    r'(no\s+|without\s+|negative\s+for\s+|denies\s+|free\s+of\s+|absence\s+of\s+)',
    re.IGNORECASE
)

def search_pattern(text, pattern, context=100):
    """
    Find all matches of `pattern` in `text`.
    Returns list of dicts: {snippet, is_negated}.
    """
    results = []
    for m in pattern.finditer(text):
        pre  = text[max(0, m.start() - context) : m.start()]
        post = text[m.end() : min(len(text), m.end() + context)]
        is_negated = bool(negation_pattern.search(pre))
        results.append({
            'snippet':    f'...{pre}[{m.group()}]{post}...',
            'is_negated': is_negated,
        })
    return results

print('Patterns compiled.')

## Search per patient

In [ ]:
records = []

for _, row in tqdm(patient_text.iterrows(), total=len(patient_text), desc='Searching notes'):
    sid  = row['subject_id']
    text = row['text']

    iv_hits   = search_pattern(text, stage_iv_pattern)
    m1_hits   = search_pattern(text, m1_pattern)
    mets_hits = search_pattern(text, mets_pattern)

    # True positive = at least one non-negated match
    has_stage_iv = any(not h['is_negated'] for h in iv_hits)
    has_m1       = any(not h['is_negated'] for h in m1_hits)
    has_mets     = any(not h['is_negated'] for h in mets_hits)

    has_negated  = (
        any(h['is_negated'] for h in iv_hits) or
        any(h['is_negated'] for h in m1_hits) or
        any(h['is_negated'] for h in mets_hits)
    )

    total_matches = (
        len([h for h in iv_hits   if not h['is_negated']]) +
        len([h for h in m1_hits   if not h['is_negated']]) +
        len([h for h in mets_hits if not h['is_negated']])
    )

    # Best sample evidence: first non-negated match across all pattern types
    sample_evidence = None
    for hits in [iv_hits, m1_hits, mets_hits]:
        non_neg = [h for h in hits if not h['is_negated']]
        if non_neg:
            sample_evidence = non_neg[0]['snippet']
            break

    records.append({
        'subject_id':          sid,
        'has_stage_iv':        has_stage_iv,
        'has_m1':              has_m1,
        'has_explicit_mets':   has_mets,
        'total_matches':       total_matches,
        'sample_evidence':     sample_evidence,
        'has_negated_matches': has_negated,
        # store snippets for sample cell
        '_iv_snippets':   [h for h in iv_hits   if not h['is_negated']],
        '_m1_snippets':   [h for h in m1_hits   if not h['is_negated']],
        '_mets_snippets': [h for h in mets_hits if not h['is_negated']],
    })

results_df = pd.DataFrame(records)
results_df['any_match'] = (
    results_df['has_stage_iv'] |
    results_df['has_m1']       |
    results_df['has_explicit_mets']
)
results_df['high_confidence'] = (
    results_df[['has_stage_iv', 'has_m1', 'has_explicit_mets']]
    .sum(axis=1) >= 2
)

print(f'Done. {len(results_df):,} patients searched.')

## Results

In [ ]:
N = len(results_df)
def fmt(n): return f'{n:>6,}  ({n/N*100:5.1f}%)'

n_iv    = results_df['has_stage_iv'].sum()
n_m1    = results_df['has_m1'].sum()
n_mets  = results_df['has_explicit_mets'].sum()
n_any   = results_df['any_match'].sum()
n_hc    = results_df['high_confidence'].sum()
n_neg   = results_df['has_negated_matches'].sum()

print('=' * 60)
print(f'  Search cohort (DS + RR notes)    : {N:,}')
print('=' * 60)
print(f'  Stage IV mention (non-negated)   : {fmt(n_iv)}')
print(f'  M1 mention (non-negated)         : {fmt(n_m1)}')
print(f'  Explicit metastases (non-neg)    : {fmt(n_mets)}')
print('-' * 60)
print(f'  Union  (any signal)              : {fmt(n_any)}')
print(f'  High-confidence (≥2 signals)     : {fmt(n_hc)}')
print('-' * 60)
print(f'  Patients with negated matches    : {fmt(n_neg)}')
print('=' * 60)

## Sample verification

5 randomly selected snippets per pattern. Check for false positives — e.g. *"no stage IV disease"*, *"M1 macrophages"*, or metastases from a different primary cancer.

In [ ]:
def show_samples(results_df, snip_col, label, n=5):
    pool = results_df[results_df[snip_col].apply(lambda x: len(x) > 0)]
    sample = pool.sample(min(n, len(pool)), random_state=42)
    print(f'\n── {label} ({len(pool):,} patients with matches) — {min(n, len(pool))} samples ──')
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        snip = row[snip_col][0]['snippet']
        print(f'  [{i}] subject_id={row["subject_id"]}')
        print(f'      {snip}')

show_samples(results_df, '_iv_snippets',   'Stage IV')
show_samples(results_df, '_m1_snippets',   'M1')
show_samples(results_df, '_mets_snippets', 'Explicit metastases')

## Decision support

**Results (run 2026-06-25):**

| Signal | Patients | % |
|--------|----------|---|
| Stage IV mention (non-negated) | 341 | 22.3% |
| M1 mention (non-negated) | 39 | 2.6% |
| Explicit metastases (non-neg) | 877 | 57.4% |
| **Union (any signal)** | **938** | **61.4%** |
| **High-confidence (≥2 signals)** | **304** | **19.9%** |
| Has negated matches | 759 | 49.7% |

**Verdict — Path A (scope to T/N only):**

The raw counts look large, but sample verification reveals two critical issues:

1. **M1 pattern is entirely false positives.** All 5 M1 samples refer to the **middle cerebral artery M1 segment** (cerebrovascular anatomy: "M1 occlusion", "M1/M2 bifurcation", "M1 branch"). This pattern cannot be used for TNM M1 staging without a much more context-specific query.

2. **Stage IV mixes cancer types.** Sample [4] explicitly shows "stage IV SCLC" (small cell, not NSCLC). The cohort includes all lung cancer patients, not NSCLC-only, so the 341 Stage IV count is inflated.

3. **Metastases mentions carry heavy negation.** 49.7% of patients have negated matches ("no evidence of metastatic disease", "no brain metastasis") — even after filtering, the remaining positives need manual label verification before use as ground truth.

Getting to reliable NSCLC Stage IV/M1 labels from these notes would require: NSCLC-type filtering, M1 pattern redesign, negation improvement, and manual review. That work was not in scope.

**Recommendation: proceed with Path A.** Document in the Guideline Logic Agent write-up that evaluation is scoped to T/N staging due to the absence of Stage IV cases in NSCLC-Radiomics and the infeasibility of reliable M1 label extraction from MIMIC-IV notes within the current project scope.

In [ ]:
output = results_df[[
    'subject_id', 'has_stage_iv', 'has_m1', 'has_explicit_mets',
    'any_match', 'high_confidence', 'total_matches',
    'has_negated_matches', 'sample_evidence'
]].copy()

out_path = PROCESSED_DIR / 'mimic_stage_iv_candidates.csv'
output.to_csv(out_path, index=False)
print(f'Saved {len(output):,} rows → {out_path}')
print(f'Candidates (any signal)     : {output["any_match"].sum():,}')
print(f'High-confidence candidates  : {output["high_confidence"].sum():,}')